# Module 50 — Synthetic Data Generation for Privacy Compliance

> **Track 4 · Marketing Analytics & Optimization**  
> **Difficulty**: Advanced  
> **Runtime**: ~10 minutes  
> **Key Libraries**: Faker, Pandas, NumPy, Plotly  
> **Prerequisite**: Module 1 (Optimized Pandas)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic User Profiles](#3-synthetic-user-profiles)
4. [Correlation Preservation](#4-correlation-preservation)
5. [Privacy Metrics](#5-privacy-metrics)
6. [Visualization](#6-visualization)
7. [Key Business Takeaway](#7-key-business-takeaway)

---
## §1 · Business Context

### Why synthetic data for privacy?

Synthetic data enables **privacy-compliant analytics**:
- **GDPR compliance**: No real personal data exposed.
- **Safe sharing**: Share data without privacy risks.
- **Testing**: Test systems with realistic but fake data.

**Applications**:
1. **Data sharing**: Share with partners safely.
2. **Development**: Test with realistic data.
3. **Analytics**: Perform analysis without exposing real users.

**Key requirements**:
- **Statistical similarity**: Preserve distributions and correlations.
- **Privacy protection**: No 1:1 mapping to real users.
- **Utility**: Useful for downstream tasks.

In this notebook we will:
1. Generate synthetic user profiles with Faker.
2. Preserve correlations between features.
3. Measure privacy metrics (uniqueness, re-identification risk).
4. Compare synthetic vs real data distributions.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Data generation ──────────────────────────────────────────────
from faker import Faker

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

fake = Faker()
Faker.seed(42)

print("✓ All imports loaded")

---
## §3 · Synthetic User Profiles

In [ ]:
# Generate synthetic user profiles
N_USERS = 10000

users = []
for i in range(N_USERS):
    users.append({
        'user_id': i,
        'name': fake.name(),
        'email': fake.email(),
        'age': np.random.normal(35, 10).clip(18, 80).astype(int),
        'income': np.random.lognormal(10.5, 0.8),
        'city': fake.city(),
        'country': fake.country_code(),
        'signup_date': fake.date_between(start_date='-2y', end_date='today'),
        'purchase_frequency': np.random.poisson(5),
        'avg_order_value': np.random.exponential(75),
    })

df_synthetic = pd.DataFrame(users)

print(f"✓ Generated {N_USERS:,} synthetic user profiles")
print(f"\nSample profiles:")
print(df_synthetic.head())

---
## §4 · Correlation Preservation

In [ ]:
# Add correlated features
# Income and purchase frequency should be correlated
df_synthetic['purchase_frequency'] = (
    0.3 * (df_synthetic['income'] / df_synthetic['income'].max()) * 10 +
    np.random.poisson(3, N_USERS)
).astype(int)

# Age and avg_order_value correlation
df_synthetic['avg_order_value'] = (
    50 + 0.5 * df_synthetic['age'] +
    np.random.exponential(30, N_USERS)
)

# Calculate CLV
df_synthetic['clv'] = (
    df_synthetic['purchase_frequency'] *
    df_synthetic['avg_order_value'] *
    0.3  # 30% margin
)

print(f"✓ Correlations added")
print(f"\nCorrelation matrix:")
corr_cols = ['age', 'income', 'purchase_frequency', 'avg_order_value', 'clv']
print(df_synthetic[corr_cols].corr().round(2))

---
## §5 · Privacy Metrics

In [ ]:
# Calculate privacy metrics
def calculate_privacy_metrics(df):
    """Calculate privacy metrics for synthetic data."""
    metrics = {}
    
    # Uniqueness (how many unique combinations)
    quasi_identifiers = ['age', 'city', 'country']
    unique_combinations = df[quasi_identifiers].drop_duplicates()
    metrics['uniqueness'] = len(unique_combinations) / len(df)
    
    # Re-identification risk (average group size)
    group_sizes = df.groupby(quasi_identifiers).size()
    metrics['avg_group_size'] = group_sizes.mean()
    metrics['min_group_size'] = group_sizes.min()
    
    # K-anonymity (minimum group size)
    metrics['k_anonymity'] = group_sizes.min()
    
    return metrics

privacy_metrics = calculate_privacy_metrics(df_synthetic)

print(f"✓ Privacy metrics calculated")
print(f"\nPrivacy Metrics:")
for metric, value in privacy_metrics.items():
    print(f"  {metric}: {value:.2f}")

print(f"\nInterpretation:")
print(f"  - K-anonymity > 5: Good privacy protection")
print(f"  - Uniqueness < 0.8: Low re-identification risk")

---
## §6 · Visualization

In [ ]:
# Compare distributions
fig = make_subplots(rows=1, cols=2, subplot_titles=['Age Distribution', 'Income Distribution'])

# Age distribution
fig.add_trace(go.Histogram(
    x=df_synthetic['age'],
    nbinsx=30,
    name='Synthetic Age'
), row=1, col=1)

# Income distribution
fig.add_trace(go.Histogram(
    x=df_synthetic['income'],
    nbinsx=50,
    name='Synthetic Income'
), row=1, col=2)

fig.update_layout(height=400, showlegend=True)
fig.show()

print("💡 Key insights:")
print("   - Synthetic data preserves realistic distributions")
print("   - No real personal data is exposed")
print("   - Safe for sharing and testing")

---
## §7 · Key Business Takeaway

> ### 🎯 Action Items for Data & Compliance Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Data sharing, development, testing, analytics |
> | **Method** | Faker for PII, copulas for correlated features |
> | **Privacy** | K-anonymity > 5, uniqueness < 0.8 |
> | **Validation** | Compare distributions, correlation preservation |
> | **Compliance** | GDPR, CCPA, internal privacy policies |
>
> ### 💡 Synthetic Data Use Cases
>
> | Use Case | Privacy Level | Data Quality |
> |----------|---------------|--------------|
> | External sharing | High | Medium |
> | Development/Testing | Medium | High |
> | Internal analytics | Low | Very High |
>
> ### 🔑 Key Takeaways
>
> 1. **Synthetic data enables privacy-compliant analytics**.
> 2. **Preserve correlations** for realistic synthetic data.
> 3. **Measure privacy metrics** (k-anonymity, uniqueness).
> 4. **Validate utility** by comparing distributions.
> 5. **Use for GDPR/CCPA compliance** in data sharing.